%% [markdown]<br>
<br>
# The Sims 4 review analysis<br>
<br>
This notebook walks through the review analysis utilities one step at a time:<br>
loading processed reviews, preprocessing text, counting words, scoring sentiment and<br>
plotting each result directly in the cell that computes it.<br>
<br>
Change `GAME` and `CUTOFF` below to analyse another processed game or move the<br>
before/after boundary. The default preset keeps the Sims-specific keywords and<br>
replacements from the original notebook.

%%

In [44]:
import matplotlib.pyplot as plt
import pandas as pd

%%

In [45]:
from steam_reviews.analysis.dataset import (
    add_period,
    add_year,
    game_name,
    load_reviews,
    split_by_cutoff,
)
from steam_reviews.analysis.plotting import (
    plot_boxplot,
    plot_distribution_fits,
    plot_engagement_boxplot,
    plot_scatter,
    plot_sentiment_categories,
    plot_sentiment_distribution,
    plot_sentiment_vs_recommendation,
    plot_word_frequencies,
    plot_wordcloud,
    plot_yearly_sentiment,
)
from steam_reviews.analysis.presets import preset_for
from steam_reviews.analysis.sentiment import add_sentiment, sentiment_summary
from steam_reviews.analysis.statistics import (
    engagement_split,
    pearson_correlations,
    yearly_mean,
)
from steam_reviews.analysis.text import (
    DEFAULT_EXTRA_STOPWORDS,
    PreprocessingConfig,
    prepare_documents,
    word_frequencies,
)

%%

In [46]:
pd.set_option("display.max_colwidth", 120)

%%

In [47]:
PRESET = preset_for("SIMS")
GAME = 730
CUTOFF = PRESET.cutoff

%%

In [48]:
def show(figure):
    """Display a figure in the notebook and release it afterwards."""
    plt.show()
    plt.close(figure)

%% [markdown]<br>
<br>
## Load processed reviews<br>
<br>
`load_reviews` reads `data/processed/reviews.csv`, selects one application ID and sorts<br>
the rows by date.

%%

In [49]:
reviews = load_reviews(GAME)

%%

In [50]:
print(f"Game: {game_name(reviews)} (app {GAME})")
print(f"Reviews: {len(reviews)}")
print(f"Date range: {reviews['date'].min().date()} to {reviews['date'].max().date()}")
print(f"Recommendation counts: {reviews['recommendation'].value_counts().to_dict()}")

Game: 730 (app 730)
Reviews: 1000
Date range: 2012-08-24 to 2026-10-07
Recommendation counts: {'Recommended': 803, 'Not Recommended': 197}


%%

In [51]:
reviews[["date", "text", "hours_played", "recommendation", "helpful_votes"]].head(10)

,date,text,hours_played,recommendation,helpful_votes
0,2012-08-24,"I have been playing Counter Strike since CS 1.6, and this game has given me so many fun memories over the years. Her...",5872.1,Recommended,89
1,2013-01-22,"Takes time to get used to, but despite all the cheaters, probably the best of whole Counter-Strike franchise. UPDATE...",1521.9,Not Recommended,165
2,2013-09-29,INCREDBILE! Over 9000!!,12810.3,Recommended,72
3,2013-11-25,"CSGO is a near-perfect multiplayer shooter. Unlike the previous versions, which were often difficult to get in to, t...",5389.7,Recommended,4606
4,2013-11-25,"Must buy if you love FPS games! This game is all about personal progression, since it comes with a high learning cur...",12642.4,Recommended,2411
5,2013-11-30,"Maybe the only FPS where it truly boils down to a game of pure skill. No bullsh*t perks or kill streaks, no auto-tur...",56.7,Recommended,250
6,2013-12-04,YEAHHHHHHHHHH ~,13238.9,Recommended,43
7,2013-12-05,"Ruined my life, 5 Stars.",1444.9,Recommended,4841
8,2013-12-14,I purchased this game on release day and played it for about three hours. It was absolute crap and I uninstalled it....,1569.0,Recommended,149
9,2013-12-16,The game of a generation. Travel the globe and gather the most wonderful objects created by man. Put them in a magic...,105128.2,Recommended,60


%% [markdown]<br>
<br>
## Preprocess text<br>
<br>
The preset preserves domain words such as `sims`, `dlc` and `buildmode`, and normalizes<br>
variants such as `pack`, `packs` and `expansion` to `dlc`. The filtered `document`<br>
column keeps only tokens that occur at least twice.

%%

In [52]:
prepared = prepare_documents(reviews, PRESET.preprocessing)

[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jgrzyb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jgrzyb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jgrzyb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]  

%%

In [53]:
print(f"Documents: {len(prepared.reviews)}")
print(f"Vocabulary: {len(prepared.vocabulary)} terms")
print(f"Sample vocabulary: {sorted(prepared.vocabulary)[:20]}")

Documents: 981
Vocabulary: 2968 terms
Sample vocabulary: ['100ms', '10k', '10x', '11k', '128tick', '144hz', '15k', '1700s', '1k', '1v1', '1v2', '1v5', '20k', '2k', '2x', '3k', '3rd', '4am', '4k', '50p']


%%

In [54]:
prepared.reviews[["text", "processed_review", "document"]].head(10)

,text,processed_review,document
0,"I have been playing Counter Strike since CS 1.6, and this game has given me so many fun memories over the years. Her...",play counter strike since c game give many fun memory year new chapter counter strike global offensive,play counter strike since c game give many fun memory year new chapter counter strike global offensive
1,"Takes time to get used to, but despite all the cheaters, probably the best of whole Counter-Strike franchise. UPDATE...",take time get use despite cheat probably best whole counter strike franchise update get ban favorite game beta overw...,take time get use despite cheat probably best whole counter strike franchise update get ban favorite game beta overw...
2,INCREDBILE! Over 9000!!,incredbile,
3,"CSGO is a near-perfect multiplayer shooter. Unlike the previous versions, which were often difficult to get in to, t...",csgo near perfect multiplayer shooter unlike previous version often difficult get one offer variety mode appeal skil...,csgo near perfect multiplayer shooter unlike previous version often difficult get one offer variety mode appeal skil...
4,"Must buy if you love FPS games! This game is all about personal progression, since it comes with a high learning cur...",must buy love fps game game personal progression since come high learning curve unlock stupid grip scope gun learn g...,must buy love fps game game personal progression since come high learning curve unlock stupid grip scope gun learn g...
5,"Maybe the only FPS where it truly boils down to a game of pure skill. No bullsh*t perks or kill streaks, no auto-tur...",maybe fps truly boil game pure skill bullsh perk kill streaks auto turrets air strike want win counter strike learn ...,maybe fps truly boil game pure skill perk kill auto air strike want win counter strike learn map play smart fast ref...
6,YEAHHHHHHHHHH ~,yeahhhhhhhhhh,
7,"Ruined my life, 5 Stars.",ruin life star,ruin life star
8,I purchased this game on release day and played it for about three hours. It was absolute crap and I uninstalled it....,purchase game release day play three hour absolute crap uninstalled fast forward one year friend recommend try well ...,purchase game release day play three hour absolute crap uninstalled fast forward one year friend recommend try well ...
9,The game of a generation. Travel the globe and gather the most wonderful objects created by man. Put them in a magic...,game generation travel globe gather wonderful object create man put magic bag pile moondust content say bag still su...,game generation travel globe gather wonderful object create man put magic bag pile content say bag still suck compar...


%% [markdown]<br>
<br>
## Word frequencies<br>
<br>
Word frequencies are computed for the whole corpus and for the two periods.<br>
`top_n` controls how many words are returned.

%%

In [55]:
before_documents, after_documents = split_by_cutoff(prepared.reviews, CUTOFF)
before_frequencies = word_frequencies(before_documents["document"], top_n=20)
after_frequencies = word_frequencies(after_documents["document"], top_n=20)

%%

In [56]:
print(f"Documents before {CUTOFF}: {len(before_documents)}")
print(f"Documents on/after {CUTOFF}: {len(after_documents)}")

Documents before 2022-10-18: 563
Documents on/after 2022-10-18: 418


%%

In [57]:
before_frequencies

,word,frequency
0,game,715
1,get,302
2,play,297
3,go,216
4,cheat,206
5,good,197
6,someone,159
7,like,149
8,people,147
9,one,145


%%

In [ ]:
show(
    plot_word_frequencies(
        before_frequencies,
        title=f"Top words before {CUTOFF}",
    )
)
show(
    plot_word_frequencies(
        after_frequencies,
        title=f"Top words on/after {CUTOFF}",
    )
)

%% [markdown]<br>
<br>
## Word clouds<br>
<br>
A word cloud is generated from the filtered documents, so frequent stopwords and<br>
one-off tokens are already removed.

%%

In [ ]:
overall_text = " ".join(prepared.reviews["document"])
show(plot_wordcloud(overall_text, title=f"Word cloud for {game_name(reviews)}"))

%%

In [ ]:
show(
    plot_wordcloud(
        " ".join(before_documents["document"]),
        title=f"Word cloud before {CUTOFF}",
    )
)
show(
    plot_wordcloud(
        " ".join(after_documents["document"]),
        title=f"Word cloud on/after {CUTOFF}",
    )
)

%% [markdown]<br>
<br>
## Sentiment scoring<br>
<br>
VADER computes a compound score between -1 and 1. `add_sentiment` also assigns the<br>
categories `Positive`, `Neutral` and `Negative`. `add_year` and `add_period` add the<br>
columns used by the later plots.

%%

In [61]:
sentiment = add_period(add_year(add_sentiment(prepared.reviews)), CUTOFF)

[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jgrzyb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


%%

In [62]:
sentiment[["date", "sentiment_score", "sentiment_category", "year", "period"]].head(10)

,date,sentiment_score,sentiment_category,year,period
0,2012-08-24,0.1779,Positive,2012,Before
1,2013-01-22,0.0289,Neutral,2013,Before
2,2013-09-29,0.0000,Neutral,2013,Before
3,2013-11-25,0.9950,Positive,2013,Before
4,2013-11-25,0.9940,Positive,2013,Before
5,2013-11-30,0.4019,Positive,2013,Before
6,2013-12-04,0.0000,Neutral,2013,Before
7,2013-12-05,-0.5859,Negative,2013,Before
8,2013-12-14,0.9887,Positive,2013,Before
9,2013-12-16,-0.1531,Negative,2013,Before


%%

In [63]:
before_sentiment, after_sentiment = split_by_cutoff(sentiment, CUTOFF)

%%

In [64]:
summary = pd.DataFrame(
    {
        "overall": pd.Series(sentiment_summary(sentiment)),
        "before": pd.Series(sentiment_summary(before_sentiment)),
        "after": pd.Series(sentiment_summary(after_sentiment)),
    }
)
summary

,overall,before,after
count,981.000000,563.000000,418.000000
mean_score,0.098840,0.126421,0.061691
positive_share,0.506626,0.507993,0.504785
neutral_share,0.134557,0.170515,0.086124
negative_share,0.358818,0.321492,0.409091


%% [markdown]<br>
<br>
## Sentiment distributions

%%

In [ ]:
show(
    plot_sentiment_distribution(
        before_sentiment["sentiment_score"],
        after_sentiment["sentiment_score"],
        cutoff=CUTOFF,
    )
)
show(
    plot_sentiment_categories(
        before_sentiment,
        after_sentiment,
        cutoff=CUTOFF,
    )
)

%% [markdown]<br>
<br>
## Sentiment over time

%%

In [66]:
yearly = yearly_mean(sentiment)
yearly

,year,sentiment_score
0,2012,0.177900
1,2013,0.140964
2,2014,0.079710
3,2015,-0.017165
4,2016,0.050819
5,2017,0.002125
6,2018,0.235691
7,2019,0.295536
8,2020,0.176221
9,2021,0.148699


%%

In [ ]:
show(plot_yearly_sentiment(yearly))

%% [markdown]<br>
<br>
## Distribution fits<br>
<br>
The fitted Normal, Laplace and rescaled Beta distributions show which shape best<br>
describes the sentiment scores in each period.

%%

In [ ]:
show(
    plot_distribution_fits(
        before_sentiment["sentiment_score"],
        title=f"Before {CUTOFF}",
    )
)
show(
    plot_distribution_fits(
        after_sentiment["sentiment_score"],
        title=f"On/after {CUTOFF}",
    )
)

%% [markdown]<br>
<br>
## Sentiment and recommendation<br>
<br>
Split reviews by the author's recommendation to see whether the text sentiment agrees<br>
with the recommendation label.

%%

In [ ]:
show(plot_sentiment_vs_recommendation(sentiment))
show(
    plot_boxplot(
        sentiment,
        category_column="recommendation",
        value_column="sentiment_score",
        title="Sentiment score by recommendation",
    )
)

%% [markdown]<br>
<br>
## Engagement and helpful votes<br>
<br>
`engagement_split` marks reviews above the median playtime. The box plot compares<br>
sentiment across recommendation and engagement groups.

%%

In [70]:
engaged = engagement_split(sentiment)
engaged[["hours_played", "helpful_votes", "engaged"]].describe()

,hours_played,helpful_votes
count,981.000000,981.000000
mean,3190.584098,688.063201
std,7028.369994,1462.218770
min,0.600000,0.000000
25%,449.500000,132.000000
50%,1436.600000,278.000000
75%,3152.400000,641.000000
max,113671.700000,20454.000000


%%

In [ ]:
show(plot_engagement_boxplot(engaged))
show(
    plot_scatter(
        sentiment,
        x_column="hours_played",
        title="Sentiment vs hours played",
    )
)
show(
    plot_scatter(
        sentiment,
        x_column="helpful_votes",
        title="Sentiment vs helpful votes",
    )
)

%% [markdown]<br>
<br>
## Correlations<br>
<br>
`pearson_correlations` compares sentiment with playtime and helpful votes before and<br>
after the cutoff.

%%

In [72]:
correlations = pearson_correlations(sentiment, CUTOFF)
correlations

,comparison,before,after
0,sentiment_score vs hours_played,-0.006325,-0.001609
1,sentiment_score vs helpful_votes,-0.094710,-0.078509


%% [markdown]<br>
<br>
## Yearly word clouds<br>
<br>
The same word-cloud helper can be applied to any subset, for example each year.

%%

In [ ]:
for year, group in sentiment.groupby("year"):
    text = " ".join(group["document"])
    if text.strip():
        show(plot_wordcloud(text, title=f"Word cloud for {year}"))

%% [markdown]<br>
<br>
## Reusing the preprocessing rules<br>
<br>
`PreprocessingConfig` can be adjusted without changing the notebook flow. This example<br>
keeps the default stopwords and adds two extra words to remove.

%%

In [74]:
custom_config = PreprocessingConfig(
    preserve_keywords=PRESET.preprocessing.preserve_keywords,
    replacements=PRESET.preprocessing.replacements,
    extra_stopwords=DEFAULT_EXTRA_STOPWORDS | {"like", "really"},
)
custom_prepared = prepare_documents(reviews, custom_config)
custom_prepared.reviews[["text", "processed_review"]].head(5)

[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jgrzyb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jgrzyb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jgrzyb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]  

,text,processed_review
0,"I have been playing Counter Strike since CS 1.6, and this game has given me so many fun memories over the years. Her...",play counter strike since c game give many fun memory year new chapter counter strike global offensive
1,"Takes time to get used to, but despite all the cheaters, probably the best of whole Counter-Strike franchise. UPDATE...",take time get use despite cheat probably best whole counter strike franchise update get ban favorite game beta overw...
2,INCREDBILE! Over 9000!!,incredbile
3,"CSGO is a near-perfect multiplayer shooter. Unlike the previous versions, which were often difficult to get in to, t...",csgo near perfect multiplayer shooter unlike previous version often difficult get one offer variety mode appeal skil...
4,"Must buy if you love FPS games! This game is all about personal progression, since it comes with a high learning cur...",must buy love fps game game personal progression since come high learning curve unlock stupid grip scope gun learn g...


%% [markdown]<br>
<br>
## Small Jev-style transformer sentiment<br>
<br>
Two optional Hugging Face decision models are available for a second opinion on<br>
sentiment. Both are non-generative and return three-class probabilities:<br>
<br>
- `nano-jev`: 22.7M parameters, the fastest option, English-only.<br>
- `laya-multilingual`: 322M parameters, slower but multilingual.<br>
<br>
Install the optional dependencies with `uv sync --extra sentiment`. Weights download<br>
from Hugging Face on first use. `MODEL_SPECS` lists the supported keys and licenses.

%%

In [75]:
from steam_reviews.analysis.transformer_sentiment import (
    DEFAULT_MODEL,
    MODEL_SPECS,
    add_transformer_sentiment,
    predict_sentiment,
    transformer_sentiment_summary,
)

%%

In [76]:
model_table = pd.DataFrame(
    [
        {
            "key": spec.key,
            "parameters": f"{spec.parameters / 1_000_000:.1f}M",
            "backend": spec.backend,
            "license": spec.license,
            "description": spec.description,
        }
        for spec in MODEL_SPECS.values()
    ]
)
model_table

,key,parameters,backend,license,description
0,nano-jev,22.7M,nanojev,Apache-2.0 with MS MARCO non-commercial note,"22.7M-parameter cross-encoder, fastest option and English-only."
1,laya-multilingual,321.9M,laya,Apache-2.0,322M-parameter multilingual decision model covering 100+ languages.
2,laya-english,421.3M,laya,Apache-2.0,"421M-parameter English Laya checkpoint, stronger on English text."
3,tiny-jev-0.6b,595.8M,tinyjev,Apache-2.0,596M-parameter Jev-style decision model trained on review sentiment.


%% [markdown]<br>
<br>
`predict_sentiment` returns the chosen label, its calibrated confidence, the full<br>
probability distribution and an expected polarity score in `[-1, 1]`.

%%

In [77]:
example_reviews = [
    "Absolutely love this game, the new update is fantastic.",
    "It is okay, nothing special, but I had some fun.",
    "Crashes constantly and the developers ignore bug reports.",
]
example_predictions = predict_sentiment(example_reviews, model=DEFAULT_MODEL)
pd.DataFrame(
    [
        {
            "review": review,
            "label": prediction.label,
            "score": round(prediction.score, 3),
            "confidence": round(prediction.confidence, 3),
            **{f"p_{label}": round(value, 3) for label, value in prediction.probabilities.items()},
        }
        for review, prediction in zip(example_reviews, example_predictions, strict=True)
    ]
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,review,label,score,confidence,p_negative,p_neutral,p_positive
0,"Absolutely love this game, the new update is fantastic.",positive,0.494,0.726,0.232,0.042,0.726
1,"It is okay, nothing special, but I had some fun.",positive,0.160,0.479,0.319,0.203,0.479
2,Crashes constantly and the developers ignore bug reports.,negative,-0.423,0.507,0.507,0.408,0.085


%% [markdown]<br>
<br>
`add_transformer_sentiment` keeps the existing VADER columns and appends<br>
`transformer_*` columns. The two scores can then be compared row by row.

%%

In [78]:
comparison = add_sentiment(prepared.reviews)
comparison = add_transformer_sentiment(comparison, model=DEFAULT_MODEL)
comparison[
    [
        "text",
        "sentiment_score",
        "sentiment_category",
        "transformer_score",
        "transformer_category",
        "transformer_confidence",
    ]
].head(10)

[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jgrzyb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


,text,sentiment_score,sentiment_category,transformer_score,transformer_category,transformer_confidence
0,"I have been playing Counter Strike since CS 1.6, and this game has given me so many fun memories over the years. Her...",0.1779,Positive,-0.475334,Negative,0.693578
1,"Takes time to get used to, but despite all the cheaters, probably the best of whole Counter-Strike franchise. UPDATE...",0.0289,Neutral,-0.782099,Negative,0.868047
2,INCREDBILE! Over 9000!!,0.0000,Neutral,-0.019827,Negative,0.372560
3,"CSGO is a near-perfect multiplayer shooter. Unlike the previous versions, which were often difficult to get in to, t...",0.9950,Positive,-0.872743,Negative,0.917726
4,"Must buy if you love FPS games! This game is all about personal progression, since it comes with a high learning cur...",0.9940,Positive,-0.864989,Negative,0.911827
5,"Maybe the only FPS where it truly boils down to a game of pure skill. No bullsh*t perks or kill streaks, no auto-tur...",0.4019,Positive,-0.474287,Negative,0.707643
6,YEAHHHHHHHHHH ~,0.0000,Neutral,0.037651,Positive,0.363407
7,"Ruined my life, 5 Stars.",-0.5859,Negative,-0.199262,Negative,0.400794
8,I purchased this game on release day and played it for about three hours. It was absolute crap and I uninstalled it....,0.9887,Positive,-0.940548,Negative,0.963019
9,The game of a generation. Travel the globe and gather the most wonderful objects created by man. Put them in a magic...,-0.1531,Negative,-0.337031,Negative,0.591380


%% [markdown]<br>
<br>
`transformer_sentiment_summary` reports the mean expected polarity and category shares.<br>
Change `model="laya-multilingual"` to try the multilingual checkpoint on the same frame.

%%

In [79]:
transformer_sentiment_summary(comparison)

{'count': 981.0,
 'mean_score': -0.4251597564950282,
 'negative_share': 0.908256880733945,
 'neutral_share': 0.010193679918450561,
 'positive_share': 0.08154943934760449}

%% [markdown]<br>
<br>
## Compare sentiment methods<br>
<br>
Use the Steam recommendation as a binary proxy label: `Recommended` maps to positive and<br>
`Not Recommended` to negative. The sample below is intentionally small because every<br>
transformer model has to score each review. Scores are cached per model under<br>
`data/processed/analysis/<game>/sentiment_cache/`, so rerunning the notebook does not<br>
repeat the slow work. The cache is cleared after each model so a 4GB GPU can run them<br>
one at a time.

%%

In [80]:
from steam_reviews.analysis.dataset import default_output_dir
from steam_reviews.analysis.sentiment_comparison import (
    compare_sentiment_methods,
    confusion_matrix,
    recommendation_truth,
)
from steam_reviews.analysis.transformer_sentiment import clear_model_cache

%%

In [81]:
SAMPLE_SIZE = 1000
MODEL_DEVICE = None
MODELS = ["nano-jev", "laya-multilingual", "laya-english", "tiny-jev-0.6b"]

In [82]:
sample = prepared.reviews.head(SAMPLE_SIZE).copy().reset_index(drop=True)
sample.insert(0, "review_id", range(len(sample)))
cache_dir = default_output_dir(game_name(reviews)) / "sentiment_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

In [83]:
def cached_scores(model):
    """Load cached transformer scores or compute and save them once."""
    path = cache_dir / f"{model}_{len(sample)}.csv"
    if path.exists():
        return pd.read_csv(path)
    scored = add_transformer_sentiment(
        sample,
        model=model,
        device=MODEL_DEVICE,
    )
    scored.to_csv(path, index=False)
    return scored

%%

In [84]:
comparison = sample[["review_id", "text", "recommendation"]].copy()
comparison["recommendation_label"] = recommendation_truth(comparison)
comparison["vader"] = add_sentiment(sample)["sentiment_category"].to_numpy()
for model in MODELS:
    comparison[model] = cached_scores(model)["transformer_category"].to_numpy()
    clear_model_cache()
comparison

[nltk_data] Downloading package wordnet to /home/jgrzyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jgrzyb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/home/jgrzyb/Documents/Python/AML_APP_Steam_Reviews/.venv/lib/python3.14/site-packages/laya/agent.py:2157: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

,review_id,text,recommendation,recommendation_label,vader,nano-jev,laya-multilingual,laya-english,tiny-jev-0.6b
0,0,"I have been playing Counter Strike since CS 1.6, and this game has given me so many fun memories over the years. Her...",Recommended,Positive,Positive,Negative,Positive,Positive,Positive
1,1,"Takes time to get used to, but despite all the cheaters, probably the best of whole Counter-Strike franchise. UPDATE...",Not Recommended,Negative,Neutral,Negative,Positive,Positive,Positive
2,2,INCREDBILE! Over 9000!!,Recommended,Positive,Neutral,Negative,Negative,Negative,Positive
3,3,"CSGO is a near-perfect multiplayer shooter. Unlike the previous versions, which were often difficult to get in to, t...",Recommended,Positive,Positive,Negative,Positive,Positive,Positive
4,4,"Must buy if you love FPS games! This game is all about personal progression, since it comes with a high learning cur...",Recommended,Positive,Positive,Negative,Positive,Positive,Positive
...,...,...,...,...,...,...,...,...,...
976,976,"It’s a great game, but Valve’s biggest weakness—even after all these updates—is that it still lacks a robust anti-ch...",Recommended,Positive,Negative,Negative,Negative,Negative,Negative
977,977,"The game is unplayable. There are cheaters in every match. Many of them have been cheating for SEVERAL YEARS, as is ...",Not Recommended,Negative,Negative,Negative,Negative,Negative,Negative
978,978,"I would like to recommend it, but cheaters that I meet every single game can't make me do it. Otherwise it's CS - th...",Not Recommended,Negative,Positive,Negative,Positive,Negative,Negative
979,979,"this game is worse than juggling your own testicles, this game has made me contemplate moving to surbia and becoming...",Recommended,Positive,Negative,Negative,Negative,Negative,Neutral


%%

In [85]:
metrics = compare_sentiment_methods(comparison, ["vader", *MODELS])
metrics

,method,reviews,coverage,neutral_rate,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,kappa,mcc
0,vader,981.0,0.865443,0.134557,0.612485,0.564166,0.595172,0.547378,0.648274,0.138491,0.156292
1,nano-jev,981.0,0.989806,0.010194,0.266735,0.560627,0.528780,0.256668,0.204551,0.024393,0.083543
2,laya-multilingual,981.0,0.993884,0.006116,0.546667,0.579243,0.622436,0.517167,0.588775,0.147377,0.197000
3,laya-english,981.0,0.773700,0.226300,0.529644,0.605804,0.642545,0.518788,0.558688,0.174032,0.245616
4,tiny-jev-0.6b,981.0,0.753313,0.246687,0.589986,0.611501,0.661959,0.564127,0.623747,0.212871,0.268765


%% [markdown]<br>
<br>
Neutral predictions stay in the confusion matrix but are excluded from accuracy and F1.<br>
`coverage` and `neutral_rate` show how often each method abstained.

%%

In [86]:
confusions = pd.concat(
    {
        method: confusion_matrix(
            comparison["recommendation_label"],
            comparison[method],
        )
        for method in ["vader", *MODELS]
    },
    names=["method", "actual"],
)
confusions

predicted                   Positive  Neutral  Negative
method            actual                               
vader             Positive       421      112       253
                  Neutral          0        0         0
                  Negative        76       20        99
nano-jev          Positive        73        8       705
                  Neutral          0        0         0
                  Negative         7        2       186
laya-multilingual Positive       387        6       393
                  Neutral          0        0         0
                  Negative        49        0       146
laya-english      Positive       258      197       331
                  Neutral          0        0         0
                  Negative        26       25       144
tiny-jev-0.6b     Positive       308      209       269
                  Neutral          0        0         0
                  Negative        34       33       128